In [1]:
import pandas as pd

# Import the cleaned economic dataset
df = pd.read_csv("../data/processed/sovereign_spreads_clean.csv")

# Sort observations by country and year
df = df.sort_values(["Country", "Year"]).reset_index(drop=True)

# Display the first five observations
df.head()

,Country,Year,Spread_bp,Debt_GDP,Fiscal_Balance,Real_GDP_Growth,Inflation,Unemployment,Current_Account,Interest_Payments,Primary_Balance
0,AT,2000,29.541667,66.6,-2.4,3.2,2.0,3.883333,-0.7,3.6,1.2
1,AT,2001,26.727500,67.2,-0.7,1.3,2.3,4.008333,-0.8,3.6,2.9
2,AT,2002,16.008333,67.4,-1.4,1.5,1.7,4.391667,2.1,3.5,2.1
3,AT,2003,4.691667,66.4,-1.8,1.1,1.3,4.791667,1.6,3.2,1.4
4,AT,2004,5.658333,65.9,-4.9,2.6,2.0,5.875000,2.1,3.1,-1.8


In [2]:
# Create the previous year's debt-to-GDP ratio for each country
df["Debt_GDP_lag1"] = df.groupby("Country")["Debt_GDP"].shift(1)

# Display the first 10 observations
df[["Country", "Year", "Debt_GDP", "Debt_GDP_lag1"]].head(10)

,Country,Year,Debt_GDP,Debt_GDP_lag1
0,AT,2000,66.6,NaN
1,AT,2001,67.2,66.6
2,AT,2002,67.4,67.2
3,AT,2003,66.4,67.4
4,AT,2004,65.9,66.4
5,AT,2005,69.4,65.9
6,AT,2006,68.2,69.4
7,AT,2007,65.8,68.2
8,AT,2008,69.5,65.8
9,AT,2009,80.8,69.5


In [3]:
# List of macroeconomic variables used for prediction
features = [
    "Debt_GDP",
    "Fiscal_Balance",
    "Real_GDP_Growth",
    "Inflation",
    "Unemployment",
    "Current_Account",
    "Interest_Payments"
]

# Create the previous year's value for each variable
for variable in features:
    df[variable + "_lag1"] = df.groupby("Country")[variable].shift(1)

# Display the first five observations
df[["Country", "Year"] + [v + "_lag1" for v in features]].head()

,Country,Year,Debt_GDP_lag1,Fiscal_Balance_lag1,Real_GDP_Growth_lag1,Inflation_lag1,Unemployment_lag1,Current_Account_lag1,Interest_Payments_lag1
0,AT,2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AT,2001,66.6,-2.4,3.2,2.0,3.883333,-0.7,3.6
2,AT,2002,67.2,-0.7,1.3,2.3,4.008333,-0.8,3.6
3,AT,2003,67.4,-1.4,1.5,1.7,4.391667,2.1,3.5
4,AT,2004,66.4,-1.8,1.1,1.3,4.791667,1.6,3.2


In [4]:
# Create the previous year's sovereign spread for each country
df["Spread_bp_lag1"] = df.groupby("Country")["Spread_bp"].shift(1)

# Display the first five observations
df[["Country", "Year", "Spread_bp", "Spread_bp_lag1"]].head()

,Country,Year,Spread_bp,Spread_bp_lag1
0,AT,2000,29.541667,NaN
1,AT,2001,26.727500,29.541667
2,AT,2002,16.008333,26.727500
3,AT,2003,4.691667,16.008333
4,AT,2004,5.658333,4.691667


In [5]:
# Create a separate dataset for forecasting
df_model = df.dropna(
    subset=[v + "_lag1" for v in features] + ["Spread_bp_lag1"]
).copy()

# Check the dimensions of the modelling dataset
df_model.shape

(225, 19)

In [6]:
# Split observations chronologically
train = df_model[df_model["Year"] <= 2019].copy()
test = df_model[df_model["Year"] >= 2020].copy()

# Display the number of observations in each group
print("Training observations:", len(train))
print("Testing observations:", len(test))

Training observations: 171
Testing observations: 54


In [7]:
# Select the lagged macroeconomic variables
lagged_features = [v + "_lag1" for v in features]

# Explanatory variables (X)
X_train = train[lagged_features]
X_test = test[lagged_features]

# Variable we want to predict (y)
y_train = train["Spread_bp"]
y_test = test["Spread_bp"]

# Check the dimensions
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (171, 7)
X_test: (54, 7)
y_train: (171,)
y_test: (54,)


In [9]:
# Save the prepared dataset for the modelling notebook
df_model.to_csv(
    "../data/processed/sovereign_spreads_model.csv",
    index=False
)